<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 16. Tiny Transformer 학습 실습 — corpus에서 생성까지

이 랩은 15번에서 확인한 decoder-only Transformer를 실제로 학습시키는 작은 현업형 루프입니다.
왼쪽에는 exercise, 오른쪽에는 solution을 띄우고 tokenizer → dataset → teacher forcing →
optimizer/scheduler → AMP → accumulation/clipping → validation/perplexity → checkpoint → 생성
순으로 따라 치세요.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 실행 규모와 완료 기준

기본 설정은 `D=64, heads=4, layers=2, context=64`로 CPU에서도 수분 이내이며 CUDA가 있으면
자동으로 AMP를 사용합니다. RTX 3080에서는 구조를 이해한 뒤 `model_dim=128`, `layers=4`,
`batch_size=64` 정도로 늘려 비교해 보세요. 큰 설정은 VRAM보다 먼저 데이터가 병목이 됩니다.

완료 기준은 validation perplexity 계산, best checkpoint 재로딩, greedy/top-k 생성까지입니다.

In [ ]:
# 준비 셀: 그대로 실행합니다.
import math
import random
import tempfile
from contextlib import nullcontext
from dataclasses import asdict
from pathlib import Path

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset

from llm_engineering_lab.transformer import (
    TinyCausalTransformer,
    TinyTransformerConfig,
)
from llm_engineering_lab.acceleration import get_accelerator

SEED = 16
random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print(ACCELERATOR.summary())

## 1) 작은 corpus 준비

인터넷 다운로드 없이 반복 실행할 수 있는 짧은 한국어 corpus입니다. 작은 데이터는 좋은 문장을
만들기보다 **파이프라인 검증**에 적합합니다. train/validation 문장을 나누되 tokenizer 문자는
전체 목록으로 구성해 `<unk>` 처리도 실습할 수 있게 합니다.

In [ ]:
corpus = [
    "모델은 데이터에서 패턴을 배운다.",
    "학습 루프는 손실을 줄인다.",
    "어텐션은 토큰 사이의 관계를 찾는다.",
    "트랜스포머는 문맥을 병렬로 처리한다.",
    "옵티마이저는 가중치를 갱신한다.",
    "검증 데이터는 일반화를 확인한다.",
    "그라디언트는 역전파로 계산한다.",
    "체크포인트는 좋은 모델을 저장한다.",
    "언어 모델은 다음 토큰을 예측한다.",
    "마스크는 미래 정보를 가린다.",
    "임베딩은 토큰을 벡터로 바꾼다.",
    "생성은 한 토큰씩 반복한다.",
    "퍼플렉서티는 예측의 불확실성을 나타낸다.",
    "탑케이 샘플링은 후보를 제한한다.",
    "작은 실험으로 코드를 먼저 검증한다.",
    "재현성을 위해 시드를 기록한다.",
]
train_documents = corpus[:12]
validation_documents = corpus[12:]
train_text = "\n".join(train_documents * 6)
validation_text = "\n".join(validation_documents * 3)
print(len(train_text), len(validation_text), train_text[:45])

## 2) Character tokenizer

BPE 대신 문자 tokenizer를 쓰면 외부 라이브러리 없이 encode/decode와 special token 역할을
완전히 볼 수 있습니다. `<pad>`는 batch 정렬, `<bos>/<eos>`는 문장 경계, `<unk>`는 사전에
없는 문자를 담당합니다. 실무에서는 tokenizer도 model artifact와 함께 버전 관리합니다.

In [ ]:
class CharTokenizer:
    def __init__(self, texts: list[str]):
        # TODO 16-1: special token + 정렬된 문자로 stoi/itos를 만드세요.
        raise NotImplementedError("TODO 16-1")

    def encode(self, text: str, *, boundaries: bool = False) -> list[int]:
        # TODO: unknown fallback과 선택적 BOS/EOS를 구현하세요.
        raise NotImplementedError("TODO 16-1")

    def decode(self, ids: list[int], *, skip_special: bool = True) -> str:
        # TODO: id를 문자로 복원하고 필요하면 special token을 제거하세요.
        raise NotImplementedError("TODO 16-1")

In [ ]:
tokenizer = CharTokenizer(corpus)
sample_ids = tokenizer.encode("모델은 학습한다.", boundaries=True)
assert sample_ids[0] == tokenizer.bos_id and sample_ids[-1] == tokenizer.eos_id
assert tokenizer.decode(sample_ids) == "모델은 학습한다."
print("vocab size:", len(tokenizer), "round trip:", tokenizer.decode(sample_ids))

## 3) Sliding-window language-model dataset

긴 token stream을 `block_size` 길이의 창으로 자릅니다. 마지막 창은 pad하고 labels는 input을
복사합니다. 모델 내부에서 `logits[:, :-1]`과 `labels[:, 1:]`를 맞추므로 한 샘플 안에
최소 두 token이 필요합니다. `stride < block_size`이면 겹치는 문맥을 더 많이 학습합니다.

In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, token_ids: list[int], block_size: int, stride: int, pad_id: int):
        # TODO 16-2: 입력 검사와 window 시작 위치를 저장하세요.
        raise NotImplementedError("TODO 16-2")

    def __len__(self) -> int:
        raise NotImplementedError("TODO 16-2")

    def __getitem__(self, index: int) -> dict[str, Tensor]:
        # TODO: block_size만큼 자르고 pad한 input/mask/labels를 반환하세요.
        raise NotImplementedError("TODO 16-2")

In [ ]:
block_size = 64
train_ids = tokenizer.encode(train_text, boundaries=True)
validation_ids = tokenizer.encode(validation_text, boundaries=True)
train_dataset = NextTokenDataset(
    train_ids, block_size, stride=32, pad_id=tokenizer.pad_id
)
validation_dataset = NextTokenDataset(
    validation_ids, block_size, stride=64, pad_id=tokenizer.pad_id
)
train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, pin_memory=ACCELERATOR.pin_memory
)
validation_loader = DataLoader(
    validation_dataset, batch_size=16, shuffle=False, pin_memory=ACCELERATOR.pin_memory
)
first_batch = next(iter(train_loader))
assert first_batch["input_ids"].shape[1] == block_size
print(
    "windows:",
    len(train_dataset),
    len(validation_dataset),
    "batch:",
    first_batch["input_ids"].shape,
)

## 4) Teacher forcing의 shift 확인

학습 시 정답 prefix를 입력으로 주고 각 위치가 바로 다음 token을 예측하게 합니다. 예를 들어
`모 → 델`, `델 → 은`처럼 입력과 target을 한 칸 어긋나게 비교합니다. 생성 단계에서는 정답
prefix가 없으므로 모델이 방금 생성한 token을 다시 입력하는 autoregressive loop가 필요합니다.

In [ ]:
preview = first_batch["input_ids"][0, :12]
teacher_inputs = preview[:-1]
next_token_targets = preview[1:]
print("input :", tokenizer.decode(teacher_inputs.tolist(), skip_special=False))
print("target:", tokenizer.decode(next_token_targets.tolist(), skip_special=False))
assert torch.equal(teacher_inputs[1:], next_token_targets[:-1])

## 5) 작은 causal Transformer 구성

15번 구조와 동일한 프로젝트 구현을 사용해 이번에는 학습 루프에 집중합니다. parameter 수를
기록하면 실험 규모와 checkpoint 크기를 추적하기 쉽습니다. `max_seq_length`는 dataset의
`block_size` 이상이어야 합니다.

In [ ]:
# TODO 16-3: 아래 크기로 config와 model을 만들고 DEVICE로 이동하세요.
lm_config = TinyTransformerConfig(
    vocab_size=len(tokenizer),
    pad_id=tokenizer.pad_id,
    max_seq_length=block_size,
    model_dim=64,
    num_heads=4,
    num_layers=2,
    feed_forward_dim=128,
    dropout=0.1,
)
model = ...
parameter_count = ...

## 6) AdamW + warmup/cosine scheduler

Warmup은 초기 불안정한 update를 줄이고, 이후 cosine decay로 learning rate를 낮춥니다.
Scheduler는 **optimizer update 직후 한 번** 호출합니다. Gradient accumulation 중 micro-batch마다
호출하면 계획보다 너무 빨리 감소하므로 update step 수를 기준으로 계산해야 합니다.

In [ ]:
epochs = 3
accumulation_steps = 2
updates_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
total_update_steps = epochs * updates_per_epoch
warmup_steps = max(1, total_update_steps // 10)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)


def lr_multiplier(step: int) -> float:
    # TODO 16-4: warmup 후 cosine decay가 되도록 0~1 배율을 반환하세요.
    raise NotImplementedError("TODO 16-4")


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)

## 7) AMP — CUDA에서만 mixed precision

RTX GPU에서는 autocast가 일부 연산을 FP16으로 수행해 속도와 메모리를 개선할 수 있습니다.
`GradScaler`는 작은 gradient가 underflow되는 것을 완화합니다. CPU에서는 `nullcontext`와
disabled scaler를 사용해 동일한 학습 함수가 그대로 동작하게 만듭니다.

In [ ]:
amp_enabled = ACCELERATOR.amp_enabled
scaler = ACCELERATOR.grad_scaler()


def amp_context():
    # TODO 16-5: CUDA면 float16 autocast, 아니면 nullcontext를 반환하세요.
    raise NotImplementedError("TODO 16-5")

## 8) Gradient accumulation + clipping 학습 함수

Accumulation은 여러 micro-batch의 gradient를 더해 큰 effective batch를 흉내 냅니다. Loss를
`accumulation_steps`로 나눠 scale을 유지하고, 마지막 남은 micro-batch도 update해야 합니다.
AMP 사용 시 `unscale_` 후 clipping하여 **실제 gradient norm**을 기준으로 제한합니다.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    accumulation_steps: int,
    clip_norm: float,
) -> dict[str, float]:
    # TODO 16-6: AMP, accumulation, unscale, clipping, optimizer/scheduler step을 구현하세요.
    raise NotImplementedError("TODO 16-6")

## 9) Validation loss와 perplexity

Validation에서는 `eval()`과 `inference_mode()`를 사용하고 optimizer를 건드리지 않습니다.
Batch별 loss 평균이 아니라 **유효 target token 수로 가중 평균**해야 마지막 padding batch 때문에
지표가 왜곡되지 않습니다. Perplexity는 `exp(cross_entropy)`이며 낮을수록 좋습니다.

In [ ]:
@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    # TODO 16-7: token-weighted validation loss와 perplexity를 계산하세요.
    raise NotImplementedError("TODO 16-7")

## 10) Best checkpoint를 저장하는 학습 loop

실제 프로젝트에서는 model뿐 아니라 config, optimizer, scheduler, epoch, metric, tokenizer 정보를
함께 저장해야 재현할 수 있습니다. 이 랩은 작업 폴더를 어지럽히지 않도록 임시 디렉터리에
checkpoint를 만들고, 뒤에서 실제로 재로딩한 후 삭제합니다.

In [ ]:
checkpoint_path = (
    Path(tempfile.gettempdir()) / "ai_lab16_best_tiny_transformer_exercise.pt"
)
history: list[dict[str, float]] = []
best_validation_loss = float("inf")

# TODO 16-8: epochs 동안 train/evaluate하고 best validation checkpoint를 저장하세요.
raise NotImplementedError("TODO 16-8")

In [ ]:
# 지표/파일 검증: loss가 반드시 단조 감소할 필요는 없고 best가 저장되면 됩니다.
assert len(history) == epochs
assert all(math.isfinite(row["validation_ppl"]) for row in history)
assert checkpoint_path.exists() and checkpoint_path.stat().st_size > 0
print("best validation loss:", round(best_validation_loss, 4))
print("checkpoint:", checkpoint_path, checkpoint_path.stat().st_size, "bytes")

## 11) 새 model 객체에 checkpoint 재로딩

저장 직후 같은 객체를 계속 쓰는 것만으로는 checkpoint가 온전한지 알 수 없습니다. Config에서
새 model을 만들고 state dict를 로드한 다음 validation을 다시 계산합니다. CPU/GPU 이동을 위해
`map_location=DEVICE`를 명시합니다.

In [ ]:
# TODO 16-9: checkpoint를 불러와 새 TinyCausalTransformer에 state를 적용하세요.
payload = ...
reloaded_model = ...
reloaded_metrics = ...

## 12) Greedy와 top-k 생성

Greedy는 매번 가장 큰 logit을 선택해 결정적이지만 반복이 생기기 쉽습니다. Top-k는 확률이 큰
k개 token만 남긴 뒤 sampling하여 다양성을 줍니다. Temperature가 낮을수록 보수적입니다.
이 tiny corpus 결과의 문장 품질보다 **train/eval 전환, context limit, 재현 가능한 sampling**을 보세요.

In [ ]:
def generate_text(
    model: TinyCausalTransformer,
    prompt: str,
    *,
    max_new_tokens: int = 28,
    strategy: str = "greedy",
) -> str:
    # TODO 16-10: prompt encode -> model.generate -> decode를 구현하세요.
    # greedy: temperature=0, top_k=None / top-k: temperature=0.8, top_k=8
    raise NotImplementedError("TODO 16-10")

In [ ]:
reloaded_model.eval()
torch.manual_seed(SEED)
prompt = "모델은"
greedy_text = generate_text(reloaded_model, prompt, strategy="greedy")
torch.manual_seed(SEED)
topk_text = generate_text(reloaded_model, prompt, strategy="top-k")
assert greedy_text.startswith(prompt) and topk_text.startswith(prompt)
print("greedy:", greedy_text)
print("top-k :", topk_text)

## 13) GPU/모델 규모 확장 실험

RTX 3080에서 아래를 한 번에 모두 키우지 말고 하나씩 바꾸며 `tokens/sec`, peak VRAM,
validation perplexity를 기록하세요.

- `model_dim`: 64 → 128 (head 수로 나누어져야 함)
- `num_layers`: 2 → 4
- `block_size`: 64 → 128 (attention 메모리는 길이의 제곱에 비례)
- `batch_size`: 16 → 32/64, 부족하면 accumulation 증가
- AMP on/off 비교: CUDA에서만 비교

작은 corpus를 그대로 둔 채 모델만 키우면 과적합하기 쉬우므로 train/validation gap도 보세요.

In [ ]:
# 최종 자동 점검
assert scheduler.last_epoch == total_update_steps
assert payload["model_config"]["num_heads"] == 4
assert payload["tokenizer_itos"] == tokenizer.itos
assert math.isfinite(reloaded_metrics["perplexity"])
assert len(greedy_text) >= len(prompt)
print("all training/generation checks passed")

In [ ]:
# 임시 checkpoint 정리. 영구 보관하려면 실행 전에 artifacts 경로로 바꾸세요.
checkpoint_path.unlink(missing_ok=True)
print("temporary checkpoint removed:", not checkpoint_path.exists())

## 마무리

이 랩에서 만든 것은 작은 모델이지만 현업 학습 파이프라인의 핵심 계약을 모두 포함합니다.
데이터/label shift, update 기준 scheduler, AMP, accumulation, clipping,
token-weighted validation,
재시작 가능한 checkpoint, 추론 전략을 각각 독립적으로 설명할 수 있는지 확인하세요.

다음 확장 과제는 (1) tokenizer를 BPE로 교체, (2) KV cache 사용/미사용 생성 시간 비교,
(3) validation 기반 early stopping, (4) 여러 seed의 평균/표준편차 기록입니다.